# Various Private Set Intersection Implementations
Private Set Intersection (PSI) is a cryptographic protocol that allows two parties to compute
the intersection of their private sets without revealing any other information about the sets.

This document presents various implementations of PSI, their advantages and disadvantages.

In [22]:
def load_set(path: str) -> set[str]:
    with open(path, "r") as f:
        return set(line.strip() for line in f)

# Load sets with 1000 random strings
# Sets were created using command: sort -R rockyou2000.txt | head -n 1000 > party_X.txt
party_a_set = load_set("data/party_a.txt")
party_b_set = load_set("data/party_b.txt")

intersection_size = len(party_a_set.intersection(party_b_set))
print(f"Intersection size: {intersection_size}")

Intersection size: 507


## Bloom Filter-based PSI
Description:\
Advantages:\
Disadvantages:\
Demo:

In [27]:
from src.psi_bloom_filter import Bloom, BloomFilterPSI
import time

# Initialize Bloom filters
expected_items = 1000
false_positive_rate = 0.01
party_a_bloom = Bloom(expected_items, false_positive_rate)
party_b_bloom = Bloom(expected_items, false_positive_rate)

# Add items to Bloom filters
for item_a in party_a_set:
    party_a_bloom.add(item_a)

for item_b in party_b_set:
    party_b_bloom.add(item_b)


In [28]:

# Compute intersection
start_time = time.time()
psi_bloom = BloomFilterPSI([party_a_bloom, party_b_bloom]).intersection()
end_time = time.time()
bloom_time = end_time - start_time
print(f"Intersection bloom filter computation time: {bloom_time} seconds")

# Get PSI elements from party A point of view
start_time = time.time()
psi_element_count = sum(1 for item in party_a_set if item in psi_bloom)
end_time = time.time()
psi_time = end_time - start_time
print(f"A: Intersection PSI computation time: {psi_time} seconds")
print(f"A: PSI element count: {psi_element_count}")
false_positives = psi_element_count - intersection_size
true_negatives = len(party_a_set) - psi_element_count
print(f"A: False positive rate: {false_positives / (false_positives + true_negatives) * 100}%")

# Get PSI elements from party B point of view
start_time = time.time()
psi_element_count = sum(1 for item in party_b_set if item in psi_bloom)
end_time = time.time()
psi_time = end_time - start_time
print(f"B: Intersection PSI computation time: {psi_time} seconds")
print(f"B: PSI element count: {psi_element_count}")
false_positives = psi_element_count - intersection_size
true_negatives = len(party_b_set) - psi_element_count
print(f"B: False positive rate: {false_positives / (false_positives + true_negatives) * 100}%")


Intersection bloom filter computation time: 8.511543273925781e-05 seconds
A: Intersection PSI computation time: 0.00023102760314941406 seconds
A: PSI element count: 512
A: False positive rate: 1.0141987829614605%
B: Intersection PSI computation time: 0.00021409988403320312 seconds
B: PSI element count: 509
B: False positive rate: 0.4056795131845842%
